# Stage 2: Data Ingestion

**Financial Forecasting & Variance Analysis** | Stage 2 of 6 | ← [Stage 1: Data Sourcing](../01-data-sourcing/01-data-sourcing.ipynb) | [Project README](../../README.md) | [Stage 3: Data Cleaning & Transformation](../03-data-cleaning-and-transformation/03-data-cleaning-and-transformation.ipynb) →

> Pull the five source files from OneDrive once and land them unchanged in the bronze layer, reconciling row counts against the published source.

| Reads | Writes |
| --- | --- |
| OneDrive source folder (via `onedrive_loader.load`) | `data/bronze/*.parquet` (gitignored) |

## 2.1 Bronze-layer principles

- **Raw values are kept as published.** Every column is stored as text (`dtype=str`), so a currency string like `$-77.00` or a date like `09/2002` arrives exactly as it was. If something looks wrong later, bronze shows whether the problem is in the source or in our cleaning.
- **Minimal reshaping only.** The two JSON files are dictionaries, so they become two-column tables (`mcc`, `description` and `transaction_id`, `target`). Nothing else changes.
- **One pull, then cache.** This is the only stage that contacts OneDrive. Stages 3–6 read the local Parquet cache, so they can be re-run without downloading anything.
- **Memory.** `transactions_data.csv` is about 1.2 GB (about 13.3 million rows). It is parsed with the pyarrow CSV engine, and each table is written and released before the next one loads.

The shared code is in [`python/ingestion.py`](../../python/ingestion.py).

In [1]:
import sys
from pathlib import Path

# Locate the repo root (the folder that holds python/config.py) and import the shared modules
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "python" / "config.py").exists())
sys.path.insert(0, str(ROOT / "python"))

import pandas as pd
import config

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

## 2.2 Load from OneDrive into bronze

The first call takes a few minutes: most of the time goes on transferring the transactions file over the Graph API.

In [2]:
from ingestion import ingest_all, read_bronze

load_log = ingest_all()
load_log

,table,source_file,rows,expected_rows,columns,action,seconds,parquet_mb,rows_match
0,users,users_data.csv,2000,2000,14,reused cache,0.00,0.10,True
1,cards,cards_data.csv,6146,6146,13,reused cache,0.00,0.20,True
2,mcc_codes,mcc_codes.json,109,109,2,reused cache,0.00,0.00,True
3,fraud_labels,train_fraud_labels.json,8914963,8914963,2,reused cache,0.00,66.80,True
4,transactions,transactions_data.csv,13305915,13305915,12,loaded from OneDrive,160.90,261.60,True


## 2.3 Row-count reconciliation

Every table should match the row count published for the source. A mismatch isn't automatically an error (for example, the publisher may have revised the file), but it has to be explained before the numbers are relied on.

In [3]:
mismatched = load_log.loc[~load_log["rows_match"], ["table", "rows", "expected_rows"]]
empty = load_log.loc[load_log["rows"] == 0, "table"].tolist()

if empty:
    raise ValueError(f"Empty bronze tables: {empty}")
if mismatched.empty:
    print("All tables reconcile to the published row counts.")
else:
    print("Row counts that differ from the published figures (investigate before stage 3):")
    display(mismatched.assign(difference=mismatched["rows"] - mismatched["expected_rows"]))

All tables reconcile to the published row counts.


## 2.4 Bronze structure

Column inventory for each table, all stored as text. Sample rows are shown for transactions only. The cards table contains card numbers and CVVs, so it is described by column names only.

In [4]:
structure = []
for table in config.SOURCES:
    df = read_bronze(table)
    structure.append({"table": table, "rows": len(df), "columns": ", ".join(df.columns)})
    del df
pd.DataFrame(structure).style.set_properties(subset=["columns"], **{"white-space": "normal"})

,table,rows,columns
0,users,2000,"id, current_age, retirement_age, birth_year, birth_month, gender, address, latitude, longitude, per_capita_income, yearly_income, total_debt, credit_score, num_credit_cards"
1,cards,6146,"id, client_id, card_brand, card_type, card_number, expires, cvv, has_chip, num_cards_issued, credit_limit, acct_open_date, year_pin_last_changed, card_on_dark_web"
2,mcc_codes,109,"mcc, description"
3,fraud_labels,8914963,"transaction_id, target"
4,transactions,13305915,"id, date, client_id, card_id, amount, use_chip, merchant_id, merchant_city, merchant_state, zip, mcc, errors"


In [5]:
read_bronze("transactions").head()

,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
0,7475327,2010-01-01 00:01:00,1556,2972,$-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,NaN
1,7475328,2010-01-01 00:02:00,561,4575,$14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,NaN
2,7475329,2010-01-01 00:02:00,1129,102,$80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,NaN
3,7475331,2010-01-01 00:05:00,430,2860,$200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,NaN
4,7475332,2010-01-01 00:06:00,848,3915,$46.41,Swipe Transaction,13051,Harwood,MD,20776.0,5813,NaN


## 2.5 Stage outcome

The bronze cache now holds a text-only copy of all five sources, and the row counts above record whether each one reconciles.

Next: [Stage 3: Data Cleaning & Transformation](../03-data-cleaning-and-transformation/03-data-cleaning-and-transformation.ipynb)